# End to end: the Gaussian-mixture setting

This is Section 5.1 of the paper, in NumPy, with no network involved: under the linear
probability path `x_tau = (1 - tau) x_0 + tau xi`, the marginals of a Gaussian mixture stay a
mixture, so the velocity field is available in closed form and the complete reverse SDE can be
written down exactly.

Two things this notebook is for:

1. **What a real model has to supply** — one `TargetTransition` and one `NoiseSchedule`.
   Everything else is library code.
2. **How to plan a speculative run before writing a coupling.** Measure `delta`, convert it into
   an acceptance probability, and read off the speedup a rule could deliver — and what it will
   cost you in batch stragglers.

It mirrors `examples/gaussian_mixture.py`, which runs the same thing as a script.

In [1]:
import numpy as np

from specdiff import (
    DelayedDriftProposal,
    BatchedSpeculativeSampler,
    DraftTree,
    NoiseSchedule,
    SpeculativeSampler,
    TargetTransition,
    Verifier,
    VerifyResult,
    standard_sampler,
)
from specdiff.ops import standard_normal_sf
from specdiff.verifiers.rank1 import Rank1Frame

rng = np.random.default_rng(0)

## 1. The model

`means` is the Euler-Maruyama reverse step (eqs. 4-5) with the mixture drift (eq. 35), and the
schedule is eq. (37). Note the shape of `means`: it takes a *stack* of states with a step index
each, because a round hands it every internal node of the draft tree at once — and nodes at
different depths sit at different steps.

In [2]:
class MixtureReverseKernel(TargetTransition):
    """Euler-Maruyama reverse kernel for a Gaussian-mixture data distribution."""

    def __init__(self, means, scales, times, *, churn=1.0):
        super().__init__()
        self.mu = np.asarray(means, dtype=np.float64)        # (components, d)
        self.s = np.asarray(scales, dtype=np.float64)        # (components,)
        self.times = np.asarray(times, dtype=np.float64)     # (N + 1,) reverse-time grid
        self.gamma = float(times[1] - times[0])
        self.eps = float(churn)
        self.dim = self.mu.shape[1]

    def velocity(self, x, tau):
        """v_tau(x) = E[xi | x] - E[x_0 | x] for the mixture."""
        m = (1.0 - tau) * self.mu
        v = (1.0 - tau) ** 2 * self.s**2 + tau**2
        diff = x[:, None, :] - m[None, :, :]
        logw = -0.5 * (diff**2).sum(-1) / v - 0.5 * self.dim * np.log(v)
        logw -= logw.max(axis=1, keepdims=True)
        w = np.exp(logw)
        w /= w.sum(axis=1, keepdims=True)                    # posterior over components
        coef = (tau - (1.0 - tau) * self.s**2) / v
        per = coef[None, :, None] * diff - self.mu[None, :, :]
        return (w[:, :, None] * per).sum(axis=1)

    def means(self, indices_in_batch, states, steps):
        out = np.empty_like(states)
        for step in sorted(set(steps)):                      # one vectorised group per step
            idx = [i for i, s in enumerate(steps) if s == step]
            t = self.times[step]
            x = states[idx]
            drift = -(1.0 + self.eps**2) * self.velocity(x, 1.0 - t) - self.eps**2 * x / t
            out[idx] = x + self.gamma * drift
        return out


class MixtureSchedule(NoiseSchedule):
    """sigma_n = eps * sqrt(2 gamma (1 - t_n) / t_n), eq. (37)."""

    def __init__(self, times, churn):
        self.times = np.asarray(times, dtype=np.float64)
        self.gamma = float(times[1] - times[0])
        self.eps = float(churn)

    def sigma(self, step):
        t = self.times[step]
        return self.eps * float(np.sqrt(2.0 * self.gamma * (1.0 - t) / t))

In [3]:
dim, num_components, N, churn = 8, 5, 50, 1.0
times = np.linspace(0.05, 0.98, N + 1)

component_means = rng.uniform(-2, 2, size=(num_components, dim))
component_scales = rng.uniform(0.10, 0.25, size=num_components)

target = MixtureReverseKernel(component_means, component_scales, times, churn=churn)
schedule = MixtureSchedule(times, churn)

print("dim", dim, " components", num_components, " steps", N, " churn", churn)
print("sigma over the trajectory:", [round(schedule(n), 3) for n in (0, N // 4, N // 2, N - 1)])

dim 8  components 5  steps 50  churn 1.0
sigma over the trajectory: [0.841, 0.315, 0.187, 0.039]


## 2. The reference run

`standard_sampler` is the non-speculative Euler-Maruyama loop: `N` target calls, `1.00x`, and the
distributional ground truth. Its terminal samples should look like the data mixture.

In [4]:
reference = standard_sampler(target, schedule, num_steps=N)
ref = reference.sample(rng.standard_normal(dim), rng=rng)
print(ref.summary())

steps=50 target_calls=50 speedup=1.000x acceptance=0.000 drafted=50 verified=50


In [5]:
trials = 400
r = np.random.default_rng(1)
samples = np.stack([reference.sample(r.standard_normal(dim), rng=r, record=False).sample
                    for _ in range(trials)])

# Ground truth: draws from the mixture itself.
pick = r.integers(num_components, size=trials)
truth = component_means[pick] + component_scales[pick, None] * r.standard_normal((trials, dim))

direction = r.standard_normal(dim)
direction /= np.linalg.norm(direction)
qs = (0.05, 0.25, 0.5, 0.75, 0.95)
print(f"{'quantile':>10}{'sampler':>12}{'mixture':>12}")
for q, a, b in zip(qs, np.quantile(samples @ direction, qs), np.quantile(truth @ direction, qs)):
    print(f"{q:>10.2f}{a:>12.3f}{b:>12.3f}")

assign = np.linalg.norm(samples[:, None, :] - component_means[None], axis=2).argmin(axis=1)
print()
print("component occupancy (sampler):", np.bincount(assign, minlength=num_components) / trials)
print("distance to nearest component:", round(float(np.linalg.norm(
    samples - component_means[assign], axis=1).mean()), 3),
    " (mixture scale ~", round(float(component_scales.mean() * np.sqrt(dim)), 3), ")")

  quantile     sampler     mixture
      0.05      -2.130      -2.138
      0.25      -0.822      -0.928
      0.50      -0.602      -0.629
      0.75      -0.348      -0.352
      0.95       2.568       2.606

component occupancy (sampler): [0.2125 0.2025 0.1875 0.2025 0.195 ]
distance to nearest component: 0.47  (mixture scale ~ 0.471 )


## 3. Measure the headroom

Now the speculative sampler, with the delayed-drift proposal of eq. (7) and the delta probe: an
exact rule that never accepts, so the run costs the same `N` NFEs as the reference but records
the mean mismatch `delta` at every node. That single number determines every acceptance
probability in the paper.

In [6]:
class DeltaProbe(Verifier):
    """Records the mean mismatch at every node, then resamples exactly."""

    name = "delta-probe"

    def __init__(self):
        self.deltas = []

    def reset(self):
        self.deltas.clear()

    def verify(self, request):
        self.deltas.append(Rank1Frame.from_request(request).delta)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


tree = DraftTree.uniform(branching=4, lookahead=3)
probe = DeltaProbe()
sampler = SpeculativeSampler(
    target=target,
    proposal=DelayedDriftProposal(target),
    schedule=schedule,
    tree=tree,
    verifier=probe,
    num_steps=N,
    check_contract=True,
)

deltas = []
for _ in range(20):
    sampler.sample(rng.standard_normal(dim), rng=rng)
    deltas.extend(probe.deltas)
deltas = np.array(deltas)

delta = float(deltas.mean())
alpha = 2.0 * standard_normal_sf(delta / 2.0)          # eq. (16)

print(tree)
print(f"  proposal budget B        {tree.budget}")
print(f"  verification batch |I|   {len(tree.internal_nodes)}   (= B / K, leaves are never parents)")
print(f"  nodes probed             {len(deltas)}")
print(f"  mean mismatch delta      {delta:.3f}   (median {np.median(deltas):.3f})")
print(f"  implied K=1 acceptance   {alpha:.3f}    [eq. 16]")
print(f"  implied chain ceiling    {1.0 / (1.0 - alpha):.2f}x   [Appendix D.1]")

DraftTree(uniform, K=4, L=3, B=84, |I|=21)
  proposal budget B        84
  verification batch |I|   21   (= B / K, leaves are never parents)
  nodes probed             1000
  mean mismatch delta      0.403   (median 0.360)
  implied K=1 acceptance   0.840    [eq. 16]
  implied chain ceiling    6.26x   [Appendix D.1]


Important limitation: a probe that never accepts advances one step per round, so the
delayed drift is never more than one step stale. The measured `delta` is therefore a **lower
bound** — under a real coupling the drift ages across the accepted prefix and the mismatch grows.

## 4. What `delta` depends on

Two knobs move it, and both are worth knowing before you tune a tree.

In [7]:
def measure_delta(dim, churn=1.0, num_steps=N, trials=8, seed=0):
    r = np.random.default_rng(seed)
    times = np.linspace(0.05, 0.98, num_steps + 1)
    tgt = MixtureReverseKernel(
        r.uniform(-2, 2, size=(num_components, dim)),
        r.uniform(0.10, 0.25, size=num_components),
        times, churn=churn,
    )
    sched = MixtureSchedule(times, churn)
    p = DeltaProbe()
    s = SpeculativeSampler(target=tgt, proposal=DelayedDriftProposal(tgt), schedule=sched,
                           tree=DraftTree.uniform(branching=2, lookahead=3), verifier=p,
                           num_steps=num_steps)
    out = []
    for _ in range(trials):
        s.sample(r.standard_normal(dim), rng=r)
        out.extend(p.deltas)
    return float(np.mean(out))


print("delta grows roughly like sqrt(d):")
print(f"{'dim':>6}{'delta':>10}{'delta / sqrt(d)':>18}{'alpha':>9}")
for d in (2, 8, 32, 128):
    dl = measure_delta(d)
    print(f"{d:>6}{dl:>10.3f}{dl / np.sqrt(d):>18.3f}{2.0 * standard_normal_sf(dl / 2.0):>9.3f}")

delta grows roughly like sqrt(d):
   dim     delta   delta / sqrt(d)    alpha
     2     0.161             0.114    0.936


     8     0.407             0.144    0.839
    32     0.786             0.139    0.694
   128     1.762             0.156    0.378


In [8]:
print("churn moves sigma and the drift together, and not at the same rate:")
print(f"{'churn':>7}{'sigma_0':>10}{'delta':>9}{'alpha':>9}")
for eps in (0.25, 0.5, 1.0, 2.0):
    dl = measure_delta(8, churn=eps)
    s0 = MixtureSchedule(np.linspace(0.05, 0.98, N + 1), eps).sigma(0)
    print(f"{eps:>7.1f}{s0:>10.3f}{dl:>9.3f}{2.0 * standard_normal_sf(dl / 2.0):>9.3f}")

churn moves sigma and the drift together, and not at the same rate:
  churn   sigma_0    delta    alpha


    0.2     0.210    0.172    0.931
    0.5     0.420    0.203    0.919


    1.0     0.841    0.407    0.839
    2.0     1.681    1.467    0.463


`delta` goes the *wrong* way with churn here, and the reason is worth understanding: `sigma`
grows like `eps` (eq. 37) but this reverse drift grows like `eps^2` — the `-(1 + eps^2) v` and
`-eps^2 x / t` terms — so the delayed drift goes stale faster than the noise covers for it. Past
`eps ~ 2` the Euler discretisation itself becomes unstable and `delta` explodes; that is a
property of the step size, not of speculation.

Two lessons. `delta` is something you **measure on your model**, not something you can reason out
from `sigma`; and it is reported normalised by `sigma` everywhere because the raw mean gap means
nothing on its own — Remark 3 is the limiting case, where zero churn makes both kernels point
masses and speculation vacuous.

## 5. Choosing a topology under a proposal budget

`B = K + ... + K^L` (eq. 12) is what a round costs the *proposal*; `|I| = B / K` is what it costs
the *target*. `DraftTree.largest_uniform` picks the deepest `(K, L)` tree that fits a budget.

In [9]:
print(f"{'budget':>8}{'K':>4}{'L':>5}{'B':>6}{'|I|':>6}{'unused budget':>16}")
for budget in (24, 120):
    for K in (1, 2, 4, 8):
        t = DraftTree.largest_uniform(budget=budget, branching=K)
        print(f"{budget:>8}{K:>4}{t.depth:>5}{t.budget:>6}{len(t.internal_nodes):>6}"
              f"{budget - t.budget:>16}")

  budget   K    L     B   |I|   unused budget
      24   1   24    24    24               0
      24   2    3    14     7              10
      24   4    2    20     5               4
      24   8    1     8     1              16
     120   1  120   120   120               0
     120   2    5    62    31              58
     120   4    3    84    21              36
     120   8    2    72     9              48


The trade is visible in one line: at a budget of 120, `K = 1` buys `L = 120` levels of a chain
that will almost never be accepted to the end, while `K = 8` buys `L = 2` with 8 candidates per
node. What decides between them is how much a second candidate actually raises the per-node
acceptance probability — which is the question Algorithm 2 exists to answer, and the reason
`|I| = B / K` matters: a wide tree drafts a lot but verifies comparatively little.

## 6. Batching, and the straggler cost

Same run over 16 trajectories. With a probe nothing accepts, so occupancy stays at 1 and the
batched numbers are degenerate — but the *projection* below is not, and it is the number to plan
against.

In [10]:
batch = 16
batched = BatchedSpeculativeSampler(
    target=target,
    proposal=DelayedDriftProposal(target),
    schedule=schedule,
    tree=tree,
    verifier=DeltaProbe(),
    num_steps=N,
    keep_trajectories=False,
)
br = batched.sample(rng.standard_normal((batch, dim)), rng=rng)
print(br.summary())

steps=50 batch=16 target_calls=51 speedup=0.980x (isolated 1.000x, occupancy 1.00) acceptance=0.000 drafted=64896 verified=16240


In [11]:
def plan_batch(alpha, lookahead, num_steps, batch_size, trials=200, seed=0):
    """Pure arithmetic: each round a trajectory advances min(Geom(alpha), L - 1) + 1 steps,
    and the batch pays the max over its live members."""
    r = np.random.default_rng(seed)
    batched_, isolated = [], []
    for _ in range(trials):
        rounds = np.zeros(batch_size, dtype=int)
        steps = np.zeros(batch_size, dtype=int)
        while (steps < num_steps).any():
            live = steps < num_steps
            run = r.geometric(1.0 - alpha, size=batch_size) - 1
            advance = np.minimum(np.minimum(run, lookahead - 1) + 1, num_steps - steps)
            steps = np.where(live, steps + advance, steps)
            rounds += live
        batched_.append(num_steps / rounds.max())
        isolated.append(np.mean(num_steps / rounds))
    return float(np.mean(batched_)), float(np.mean(isolated))


b_speedup, i_speedup = plan_batch(alpha, tree.depth, N, batch)
print(f"at the measured alpha = {alpha:.3f}, L = {tree.depth}, N = {N}:")
print(f"  projected batched speedup  (batch={batch})  {b_speedup:.2f}x")
print(f"  projected isolated speedup               {i_speedup:.2f}x")
print(f"  straggler cost                           {100 * (1 - b_speedup / i_speedup):.1f}%")
print()
print(f"{'L':>4}{'isolated':>12}{'batch=16':>12}{'straggler':>12}")
for L in (2, 3, 4, 6, 8):
    b, i = plan_batch(alpha, L, N, batch)
    print(f"{L:>4}{i:>11.2f}x{b:>11.2f}x{100 * (1 - b / i):>11.1f}%")

at the measured alpha = 0.840, L = 3, N = 50:
  projected batched speedup  (batch=16)  2.22x
  projected isolated speedup               2.51x
  straggler cost                           11.7%

   L    isolated    batch=16   straggler
   2       1.83x       1.70x        7.0%
   3       2.51x       2.22x       11.7%
   4       3.08x       2.59x       15.9%


   6       3.97x       3.09x       22.1%
   8       4.60x       3.39x       26.3%


Deeper trees raise both numbers and widen the gap, because a round's advance becomes more
variable and the max over 16 draws grows faster than the mean.

## 7. Applying an acceptance rule

The diagnostic rule used above is exact but never accepts, so its speedup is `1.00x`. Use one
of the following couplings to enable speculative acceptance:

```
specdiff/verifiers/rmc.py
    ReflectionMaximalCoupling   Algorithm 1, K = 1
specdiff/verifiers/dgrs.py
    GreedyRejectionSampling     Algorithm 2, any K
```

The verifier tutorial explains the rank-1 coordinates, implementation steps, and exactness tests:
[`verifier_tutorial.ipynb`](verifier_tutorial.ipynb). Compare a verifier against the `delta`,
`alpha`, and projected speedups measured in this notebook:

* `report.acceptance_rate` from `check_exactness` at this `delta` should land near the `alpha`
  printed below for `K = 1`;
* a chain run should approach `1 / (1 - alpha)` and a `(K, L)` tree should beat it;
* `result.acceptance_rate` from a real run should come out *below* the probe's implied `alpha`,
  because the drift ages across accepted prefixes.

In [12]:
print(f"targets to hit once a coupling exists (dim={dim}, N={N}, churn={churn}):")
print(f"  {'measured delta (lower bound)':<30} {delta:.3f}")
print(f"  {'implied K=1 acceptance':<30} {alpha:.3f}")
print(f"  {'chain ceiling':<30} {1.0 / (1.0 - alpha):.2f}x")
print(f"  {'K=%d, L=%d projected' % (tree.branching, tree.depth):<30} {b_speedup:.2f}x at batch {batch}")

targets to hit once a coupling exists (dim=8, N=50, churn=1.0):
  measured delta (lower bound)   0.403
  implied K=1 acceptance         0.840
  chain ceiling                  6.26x
  K=4, L=3 projected             2.22x at batch 16


## Recap of the library

| notebook | component |
| --- | --- |
| [`tree_tutorial.ipynb`](tree_tutorial.ipynb) | `DraftTree` — topologies, budgets, truncation |
| [`kernels_tutorial.ipynb`](kernels_tutorial.ipynb) | `TargetTransition`, `ProposalTransition`, `NoiseSchedule` |
| [`verifier_tutorial.ipynb`](verifier_tutorial.ipynb) | `Verifier`, `Rank1Frame`, `check_exactness` |
| [`sampler_tutorial.ipynb`](sampler_tutorial.ipynb) | `SpeculativeSampler` — Algorithm 3 and its accounting |
| [`batched_tutorial.ipynb`](batched_tutorial.ipynb) | `BatchedSpeculativeSampler` — many trajectories, one call |
| [`backends_tutorial.ipynb`](backends_tutorial.ipynb) | `ops.Backend` — NumPy, torch, or your own |
| this one | a real model, end to end |